# Capa 3 — Análisis Exploratorio de Datos (EDA)
## Proyecto: Customer Analytics — "Super Compras"

**Metodología:** 6 Capas (Estrategia → Preparación → **EDA** → Análisis Clásico → ML → Entregable)

**Nivel de la pirámide:** Nivel I (Descriptivo) — este notebook explora y describe, no contrasta
hipótesis. Los patrones que aquí se detecten como "visualmente distintos" se pondrán a prueba
estadísticamente en la Capa 4 (`04_capa4_analisis_inferencial.ipynb`).

**Principio de Pareto 20/80:** no se agota cada columna con los 15 tipos de gráfico posibles —
se prioriza el 20% de vistas (univariado clave + 3-4 cruces) que explican el 80% del
comportamiento de los clientes de "Super Compras".

**Entrada:** `tablon_analitico_reconstruido.csv` (generado por `02_capa2_preparacion_datos.ipynb`).
Si no lo tienes, ejecuta primero ese notebook o sube el archivo a `/content/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RUTA = "./"   # carpeta donde está tablon_analitico_reconstruido.csv


## 0. Carga y construcción de la vista a nivel cliente

**Decisión metodológica importante:** `tablon_analitico_reconstruido.csv` tiene 54.405 filas —
una por línea de producto dentro de la ventana de 6 meses. Si se calculan histogramas o
correlaciones directamente sobre esas 54.405 filas, un cliente con 20 líneas de producto pesará
20 veces más que uno con 1 sola línea, **distorsionando artificialmente todas las distribuciones**
(pseudo-replicación). Por eso el EDA de variables de cliente (RFM, Segmento, sociodemográficas)
se hace sobre una vista deduplicada a **1 fila = 1 cliente (14.367 filas)**.

In [ ]:
t = pd.read_csv(RUTA + "tablon_analitico_reconstruido.csv", parse_dates=["Fecha", "Fecha_Nacimiento"])
print("Tablón transaccional:", t.shape)

cliente_cols = ["ID_Cliente", "MesCohorte", "Recency", "Frequency", "Monetary", "r", "f", "m",
                "RFM", "Segmento", "ltv", "Genero", "Fecha_Nacimiento", "Codigo_Postal",
                "Nivel_Educativo", "Ocupacion", "Tamano_Hogar", "CAC"]
cli = t[cliente_cols].drop_duplicates("ID_Cliente").reset_index(drop=True)
cli["Edad"] = ((pd.Timestamp("2023-12-31") - cli["Fecha_Nacimiento"]).dt.days / 365.25).astype(int)

print("Vista a nivel cliente:", cli.shape)
assert cli["ID_Cliente"].is_unique, "Hay ID_Cliente duplicados"
print("Nulos:", cli.isna().sum().sum(), "| Duplicados:", cli.duplicated().sum())
cli.head()


## A. Análisis univariado — variables cuantitativas

Variables: `Recency` (días desde la última compra), `Frequency` (nº transacciones/año),
`Monetary` (€ gastados/año), `ltv` (€ dentro de la ventana de 6 meses), `CAC` (coste de
adquisición) y `Edad` (derivada de `Fecha_Nacimiento`).

In [ ]:
num_vars = ["Recency", "Frequency", "Monetary", "ltv", "CAC", "Edad"]

resumen = pd.DataFrame({
    "media": cli[num_vars].mean(),
    "mediana": cli[num_vars].median(),
    "desv_std": cli[num_vars].std(),
    "asimetria": cli[num_vars].apply(stats.skew),
    "curtosis": cli[num_vars].apply(stats.kurtosis),
    "min": cli[num_vars].min(),
    "max": cli[num_vars].max(),
}).round(2)
resumen


**Lectura de la asimetría (`asimetria`):**
- `Frequency` (3,03) y `CAC` (3,48) son las más sesgadas a la derecha: la mayoría de clientes
  tienen valores bajos y una cola larga de pocos clientes con valores muy altos. **La media
  engaña aquí — hay que describir estos clientes con la mediana**, no con el promedio.
- `Monetary` (1,44) y `ltv` (1,59) también sesgadas a la derecha, típico del gasto en retail.
- `Recency` (−0,63) sesgada a la izquierda: hay más clientes acumulados en valores altos
  (llevan mucho tiempo sin comprar) que en valores bajos — coherente con que "En Riesgo" y
  "Dudosos" son, como se verá, los dos segmentos más numerosos.
- **`Recency` muestra además un patrón bimodal** (un grupo concentrado en 0-50 días y otro,
  más grande, en 250-350 días, con un valle entre medias): son dos poblaciones de clientes
  mezcladas — compradores activos recientes vs. clientes inactivos desde hace 8-11 meses — no
  un continuo homogéneo. Esto refuerza la lógica de segmentar en vez de tratar a la base de
  clientes como un bloque único.
- `Edad` (−0,08) es prácticamente simétrica — no requiere ninguna transformación.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, v in zip(axes.flat, num_vars):
    sns.histplot(cli[v], kde=True, ax=ax, color="#2E7D32")
    ax.axvline(cli[v].mean(), color="red", ls="--", lw=1, label="media")
    ax.axvline(cli[v].median(), color="orange", ls="--", lw=1, label="mediana")
    ax.set_title(v)
    ax.legend(fontsize=8)
plt.suptitle("Distribución de las variables cuantitativas (nivel cliente, n=14.367)", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(16, 4))
for ax, v in zip(axes, num_vars):
    sns.boxplot(y=cli[v], ax=ax, color="#81C784")
    ax.set_title(v, fontsize=10)
plt.suptitle("Boxplots — dispersión y outliers", y=1.03)
plt.tight_layout()
plt.show()


**Outliers (regla IQR: fuera de [Q1-1,5·IQR, Q3+1,5·IQR]):**

In [ ]:
for v in ["Frequency", "Monetary", "ltv", "CAC"]:
    q1, q3 = cli[v].quantile([.25, .75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = cli[(cli[v] < low) | (cli[v] > high)]
    print(f"{v:10s} límite superior IQR = {high:8.2f} -> {len(out):5d} outliers ({len(out)/len(cli):.1%})")


**Nota sobre `CAC`:** un 7,5% de clientes son outliers por IQR, con casos extremos como el
cliente con CAC = 1.426€ que solo generó 36,84€ de `Monetary` en todo el año. No son errores de
captura (son costes de adquisición de campañas específicas) — son clientes reales que costaron
mucho más de lo que han devuelto. Esto es información de negocio valiosa, no ruido a eliminar:
se retoma en la Capa 5 al cruzar CAC contra LTV. **No se recortan ni transforman estos outliers
en este proyecto** porque el propio objetivo de negocio (identificar quién no es rentable) los
necesita intactos.

### QQ-Plot — ¿son normales `Monetary` y `ltv`?

Esto es preparación directa para la Capa 4: si no hay normalidad, ahí tocará usar
Mann-Whitney/Kruskal-Wallis en vez de t-Student/ANOVA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, v in zip(axes, ["Monetary", "ltv"]):
    stats.probplot(cli[v], dist="norm", plot=ax)
    ax.set_title(f"QQ-Plot — {v}")
plt.tight_layout()
plt.show()

for v in ["Monetary", "ltv"]:
    # Kolmogorov-Smirnov (muestra grande): H0 = los datos son normales
    stat, p = stats.kstest(cli[v], "norm", args=(cli[v].mean(), cli[v].std()))
    print(f"{v:10s} K-S p-valor = {p:.2e} -> {'Normal' if p > 0.05 else 'NO normal'}")


Los puntos se alejan claramente de la diagonal en la cola superior (coherente con la
asimetría positiva ya vista) y el test K-S confirma **no-normalidad** en ambas variables — esto
ya anticipa que las comparaciones de la Capa 4 sobre `Monetary`/`ltv` probablemente necesiten
pruebas no paramétricas (Mann-Whitney / Kruskal-Wallis), a confirmar formalmente allí.

## B. Análisis univariado — variables cualitativas

`Segmento`, `Genero`, `Nivel_Educativo`, `Ocupacion`, `Tamano_Hogar` y `MesCohorte`.

In [ ]:
cat_vars = ["Segmento", "Genero", "Nivel_Educativo", "Ocupacion", "Tamano_Hogar", "MesCohorte"]

fig, axes = plt.subplots(3, 2, figsize=(13, 13))
for ax, v in zip(axes.flat, cat_vars):
    order = cli[v].value_counts().index
    sns.countplot(y=cli[v], order=order, ax=ax, color="#3b7a57")
    total = len(cli)
    for i, cat in enumerate(order):
        pct = (cli[v] == cat).mean() * 100
        ax.text(cli[v].value_counts()[cat] + total * 0.01, i, f"{pct:.1f}%", va="center", fontsize=8)
    ax.set_title(v)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


**Lecturas clave:**
- `Segmento`: **"En Riesgo" (41,4%) y "Dudosos" (40,3%) concentran el 81,7% de la base de
  clientes.** Solo un 3,9% son "Campeones". Esto es el hallazgo más importante del EDA: la
  mayoría de la cartera de "Super Compras" está en zona de riesgo o sin comportamiento claro —
  el negocio depende de muy pocos clientes de alto valor.
- `Genero`: 62,7% Mujer / 37,3% Hombre — desbalance real, a verificar en la Capa 4 si también
  se refleja en diferencias de Monetary/Segmento o es solo una diferencia de volumen de clientela.
- `Nivel_Educativo`/`Ocupacion`: recuerda que "Desconocido" es una categoría real de nulos
  imputados en la Capa 2 (9,5% y 11,6% del total respectivamente) — no es una categoría de
  negocio, hay que tratarla con cautela en cualquier interpretación de la Capa 4.
- `MesCohorte`: la cohorte de enero (3.114 clientes) es la más numerosa; julio, la más pequeña
  (1.168) — coherente con ser una ventana de captación más corta (ver Capa 2).

## C. Análisis multivariado — cuantitativa × cuantitativa

In [ ]:
corr = cli[num_vars].corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Matriz de correlación (Pearson) — nivel cliente")
plt.tight_layout()
plt.show()


**Lectura — correlación no implica causalidad, y aquí hay que ir con cuidado extra:**
- `Monetary` ↔ `ltv`: **r = 0,94.** Es esperable y NO es un hallazgo de negocio: `ltv` se
  construye a partir de `Monetary` dentro de una ventana (ver Capa 2) — son, en gran medida,
  la misma información medida dos veces. Por eso, en la Capa 5, nunca se usará `Monetary` como
  predictor de `ltv` (sería circular).
- `Frequency` ↔ `Monetary`: r = 0,62 (moderada-alta) — comprar más veces se asocia con gastar
  más en total, razonable.
- `Recency` ↔ `Frequency`: r = −0,54 — cuanto más tiempo lleva un cliente sin comprar, menos
  transacciones acumula, también razonable.
- **`CAC` prácticamente no correlaciona con nada** (r entre 0,00 y 0,14 con el resto): lo que
  costó adquirir a un cliente **no predice** cuánto va a gastar. Es un hallazgo genuino y nada
  obvio — a confirmar con un contraste formal en la Capa 4, pero de entrada sugiere que el
  presupuesto de adquisición actual no está bien dirigido hacia los clientes de más valor.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sample = cli.sample(3000, random_state=42)  # muestreo para legibilidad del scatter
sns.scatterplot(data=sample, x="Monetary", y="ltv", hue="Segmento", alpha=0.5, s=18, ax=ax,
                 palette="tab10", legend=False)
ax.plot([0, 500], [0, 500], color="black", ls="--", lw=1, label="Monetary = ltv")
ax.legend()
ax.set_title("Monetary vs ltv (muestra de 3.000 clientes)")
plt.tight_layout()
plt.show()


## D. Análisis multivariado — cualitativa × cualitativa

Tabla de contingencia `Segmento` × `Genero`, con porcentajes **condicionados por fila** (dentro
de cada segmento, qué % es Mujer/Hombre).

In [ ]:
ct_pct = pd.crosstab(cli["Segmento"], cli["Genero"], normalize="index").mul(100).round(1)
ct_n = pd.crosstab(cli["Segmento"], cli["Genero"])
ct_pct


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ct_pct.loc[cli["Segmento"].value_counts().index].plot(kind="barh", stacked=True, ax=ax,
                                                         color=["#4A90D9", "#E88AA6"])
ax.axvline(50, color="black", lw=0.8, ls=":")
ax.set_xlabel("% dentro del segmento")
ax.set_title("Composición de género por segmento")
plt.tight_layout()
plt.show()


El % de mujeres oscila entre 52,4% (Prometedores) y 69,4% (Leales Potenciales) — hay
diferencia, pero no es un patrón extremo (todos los segmentos rondan la proporción global de
62,7%/37,3%). Habrá que confirmar con Chi-cuadrado en la Capa 4 si esta variación es
estadísticamente significativa o es ruido, especialmente en los segmentos pequeños
(Prometedores tiene solo 21 clientes — cuidado con sacar conclusiones de muestras tan chicas).

## E. Análisis multivariado — cualitativa × cuantitativa

`Monetary` y `ltv` por `Segmento` (el cruce más relevante del proyecto: es la base de la
segmentación de valor que pide el negocio).

In [ ]:
orden_seg = cli.groupby("Segmento")["Monetary"].median().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=cli, x="Segmento", y="Monetary", order=orden_seg, ax=axes[0], color="#66BB6A")
axes[0].set_title("Monetary por Segmento")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=cli, x="Segmento", y="ltv", order=orden_seg, ax=axes[1], color="#4A90D9")
axes[1].set_title("ltv por Segmento")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Medias +/- IC 95% por segmento (diagrama de error, Capa 3 del checklist)
resumen_seg = cli.groupby("Segmento")["Monetary"].agg(["mean", "std", "count"])
resumen_seg["ic95"] = 1.96 * resumen_seg["std"] / np.sqrt(resumen_seg["count"])
resumen_seg = resumen_seg.sort_values("mean", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(resumen_seg["mean"], resumen_seg.index, xerr=resumen_seg["ic95"],
            fmt="o", color="#2E7D32", capsize=4)
ax.set_xlabel("Monetary medio (€) ± IC 95%")
ax.set_title("Monetary medio por Segmento, con intervalo de confianza al 95%")
plt.tight_layout()
plt.show()
resumen_seg.round(2)


Los boxplots y los intervalos de confianza muestran diferencias visualmente claras entre
segmentos (esperable, ya que el propio `Segmento` se construye a partir de `Monetary` — esto NO
es un hallazgo inferencial válido en sí mismo, es casi tautológico). Lo que sí es un contraste
legítimo para la Capa 4 es cruzar `Monetary`/`ltv` con las variables **sociodemográficas**
(Género, Nivel_Educativo, Ocupacion, Tamano_Hogar) — esas sí son independientes de cómo se
construyó el Segmento.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=cli, x="Nivel_Educativo", y="Monetary",
            order=cli.groupby("Nivel_Educativo")["Monetary"].median().sort_values(ascending=False).index,
            ax=ax, color="#F4A261")
ax.set_title("Monetary por Nivel Educativo")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

cli.groupby("Nivel_Educativo")["Monetary"].agg(["median", "mean", "count"]).round(2)


## F. Principio de Pareto aplicado a `Monetary`

¿Qué porcentaje de clientes genera el 80% de las ventas? (pregunta explícita del propio
material de negocio).

In [ ]:
s = cli.sort_values("Monetary", ascending=False)["Monetary"].values
cum_pct_ventas = np.cumsum(s) / s.sum()
n80 = int((cum_pct_ventas < 0.80).sum() + 1)
print(f"{n80} de {len(cli)} clientes ({n80/len(cli):.1%}) generan el 80% del Monetary total.")

fig, ax = plt.subplots(figsize=(6.5, 5))
x = np.arange(1, len(s) + 1) / len(s) * 100
ax.plot(x, cum_pct_ventas * 100, color="#2E7D32", lw=2)
ax.axhline(80, color="red", ls="--", lw=1)
ax.axvline(n80 / len(cli) * 100, color="red", ls="--", lw=1)
ax.set_xlabel("% de clientes (ordenados de mayor a menor gasto)")
ax.set_ylabel("% acumulado de Monetary")
ax.set_title("Curva de concentración de ventas")
plt.tight_layout()
plt.show()


**Este es un hallazgo que contradice la heurística habitual del "80/20":** en "Super
Compras" hace falta el **41,9% de los clientes** (no el 20%) para llegar al 80% de las ventas.
La cartera está menos concentrada de lo que suele asumirse por defecto — un argumento en contra
de centrar toda la estrategia de retención únicamente en los "Campeones" (solo 555 clientes,
3,9% de la base): hay un volumen grande de clientes de valor medio ("Grandes Gastadores",
"Dudosos" con gasto medio-alto) que también sostienen el negocio.

## Resumen de hallazgos y preguntas para la Capa 4

| # | Hallazgo (EDA, Nivel I) | Pregunta a contrastar en Capa 4 (Nivel II) |
|---|---|---|
| 1 | "En Riesgo" + "Dudosos" = 81,7% de la cartera | ¿Qué variables sociodemográficas distinguen a los "En Riesgo" del resto? (Chi-cuadrado) |
| 2 | `CAC` no correlaciona con `Monetary`/`ltv` (r<0,15) | ¿Es esa correlación significativamente distinta de 0? (test de correlación) |
| 3 | Genero varía entre 52-69% Mujer según segmento | ¿Segmento y Genero son independientes? (Chi-cuadrado) |
| 4 | `Monetary` no es normal (K-S, QQ-plot) | Usar Mann-Whitney/Kruskal-Wallis, no t-Student/ANOVA, al comparar grupos |
| 5 | Diferencia de `Monetary` por Nivel_Educativo (visual) | ¿Es significativa tras corregir por comparaciones múltiples? (Kruskal-Wallis + post-hoc) |
| 6 | 41,9% de clientes generan el 80% de ventas (no 20%) | Dato descriptivo, no requiere contraste — insumo directo para el Plan de Acción de la Capa 6 |
| 7 | Outliers reales en CAC (hasta 1.426€ sin retorno) | Insumo para la Capa 5 (predicción de riesgo temprano), no se recortan |

**Siguiente paso:** `04_capa4_analisis_inferencial.ipynb` — verificación formal de supuestos
(normalidad, homocedasticidad) y contraste de hipótesis sobre los puntos 1, 2, 3 y 5 de la
tabla anterior.

In [ ]:
cli.to_csv("cliente_nivel_eda.csv", index=False)
print("Guardado: cliente_nivel_eda.csv (vista a nivel cliente, con Edad calculada) para la Capa 4.")
